# Chapter 1 standalone: the Reliability Gap, measured

*Google Gemini edition*

**Author: Imran Ahmad** · Companion notebook for *Building Reliable AI-Assisted Software Systems* (Packt). All characters, companies, incidents, and data are fictional.

## Objectives

Chapter 1 argues that software has shifted from deterministic to probabilistic, and that the distance between an impressive demo and a trustworthy production system is structural. This notebook reproduces the chapter's four quantitative beats offline:

- five identical calls that refuse to produce one identical answer
- reliability multiplying down a five-step chain: 90 percent per step, 59 percent end to end
- the 80/20 Demo Trap: a flawless 12-question demo against 413 of 500 production tickets
- an agent loop whose action sequences drift run to run

Everything runs from a seeded simulation layer: no key, no network, byte-identical numbers on every run.

![timeline](assets/Figure_1_1_timeline.png)

Five decades of guaranteed determinism, then sampled probability: the timeline above is the shift the whole book answers.

### Provider edition: Google Gemini

This edition adds one optional live cell that asks a real model the same question three times, the chapter's nondeterminism beat run against a live provider. Requirements beyond the base bundle: `google-genai` (see `requirements-providers.txt`). Set `GEMINI_API_KEY` to enable the live probe. **Simulation Mode is the canonical path**: without the provider, every number in this notebook still reproduces offline and the live cell skips itself.

In [1]:
# Chapter 1 - The Reliability Gap
# Author: Imran Ahmad
# Standalone companion notebook (core). Runs fully offline; every number is canon.
from model_sim import CANON_CH1, mock_complete, run_agent, run_traffic

print("setup complete: seeded simulation layer loaded")

setup complete: seeded simulation layer loaded


## Same input, sampled output

The oldest guarantee in software engineering is that the same code on the same input gives the same output. The cell below runs one prompt five times against the seeded mock and gets a taste of what replaces that guarantee.

In [2]:
# 1. Five identical calls, more than one answer
prompt = "In one short sentence, what is the refund window?"
answers = [mock_complete(prompt, run) for run in range(CANON_CH1["CALLS"])]
for i, a in enumerate(answers, 1):
    print(f"[call {i}] {a}")
distinct = len(set(answers))
assert 1 < distinct <= CANON_CH1["CALLS"]
print(f"{CANON_CH1['CALLS']} identical calls, {distinct} distinct answers: "
      "assert actual == expected has no single 'actual' to pin")

[call 1] You can get a refund within 30 days.
[call 2] Our refund window is 30 days from the purchase date.
[call 3] Our refund window is 30 days from the purchase date.
[call 4] You have a 30-day window to request a refund.
[call 5] Purchases can be refunded up to 30 days after you buy.
5 identical calls, 4 distinct answers: assert actual == expected has no single 'actual' to pin


Four different phrasings of one correct fact. None of them is wrong, and that is the point: a string assertion has nothing to pin, which is where Chapter 3's golden datasets eventually pick up.

## Reliability compounds down a chain

One probabilistic step is manageable. Five in a row are not, because the failure rates multiply.

In [3]:
# 2. Reliability compounds down a chain
step, steps = CANON_CH1["STEP_RELIABILITY"], CANON_CH1["CHAIN_STEPS"]
chained = round(step ** steps, 4)
assert chained == CANON_CH1["CHAINED"]
print(f"{step:.0%} per step across {steps} steps = {chained:.0%} end-to-end: "
      "reliability multiplies down a chain, it does not average")

90% per step across 5 steps = 59% end-to-end: reliability multiplies down a chain, it does not average


## The 80/20 Demo Trap

A demo samples the traffic you rehearsed. Production samples everything else. The simulation below runs the same prototype against both distributions.

In [4]:
# 3. The 80/20 Demo Trap
demo_passes, _ = run_traffic("demo", CANON_CH1["DEMO_QUESTIONS"])
prod_passes, slices = run_traffic("production", CANON_CH1["PROD_TICKETS"])
print(f"demo: {demo_passes}/{CANON_CH1['DEMO_QUESTIONS']} flawless; "
      f"production: {prod_passes}/{CANON_CH1['PROD_TICKETS']} "
      f"({prod_passes * 100 / CANON_CH1['PROD_TICKETS']:.1f}%)")
for name, (ok, total) in slices.items():
    print(f"   {name:<24} {ok}/{total}")

demo: 12/12 flawless; production: 413/500 (82.6%)
   clean, common questions  280/294
   ambiguous phrasing       75/96
   multi-part tickets       32/51
   edge-case policies       20/37
   adversarial or hostile   6/22


The aggregate hides the mechanism, so read the slice table: clean common questions pass at 95 percent while adversarial traffic passes below a third. The demo only ever saw the first slice.

## Three layers of drift in a loop

Put the same model in a loop and the fan-out compounds: sampling, context growth, and tool outcomes each add randomness at every step.

![three layers](assets/Figure_1_5_three_layers.png)

The figure shows the three layers; the cell runs ten identical tasks through them.

In [5]:
# 4. A model in a loop drifts
runs = CANON_CH1["AGENT_RUNS"]
trajectories = {tuple(run_agent(r)) for r in range(runs)}
for t in sorted(trajectories)[:3]:
    print("   " + " -> ".join(t))
print(f"same task, {runs} agent runs, {len(trajectories)} distinct action sequences")

   read_ticket -> ask_clarifying -> read_ticket -> search_kb -> read_ticket -> search_kb
   read_ticket -> ask_clarifying -> search_kb -> check_policy -> search_kb -> read_ticket
   read_ticket -> ask_clarifying -> search_kb -> read_ticket -> search_kb -> draft_reply
same task, 10 agent runs, 10 distinct action sequences


## The gap, in one line

In [6]:
# 5. The gap, in one line
print(f"the gap: demo {demo_passes}/{CANON_CH1['DEMO_QUESTIONS']}, "
      f"production {prod_passes}/{CANON_CH1['PROD_TICKETS']}")

the gap: demo 12/12, production 413/500


In [7]:
# Optional live probe (Google Gemini): the canonical
# numbers above come from the seeded simulation layer in BOTH modes.
import os

PROBE_PROMPT = "In one short sentence, what is the refund window for an annual license?"

if os.getenv("GEMINI_API_KEY"):
    from google import genai

    llm = genai.Client()  # reads GEMINI_API_KEY from the environment

    def ask(prompt: str) -> str:
        reply = llm.models.generate_content(
            model="gemini-2.5-flash",  # check llm.models.list() for newer ids
            contents=prompt,
        )
        return reply.text or "(empty reply)"

    replies = [ask(PROBE_PROMPT) for _ in range(3)]
    for i, r in enumerate(replies, 1):
        print(f"[live call {i}] {r[:120]}")
    print("distinct answers from 3 identical live calls:", len(set(replies)))
else:
    print("SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.")
    print("All canonical numbers above came from the seeded simulation layer.")

SIMULATION MODE: GEMINI_API_KEY not set; live probe skipped.
All canonical numbers above came from the seeded simulation layer.


## Summary

The working rule this notebook leaves behind: a probabilistic core cannot be tested, demoed, or reasoned about like deterministic code. Five identical calls fanned out into four answers, a 90 percent step rate decayed to 59 percent across a chain, and the demo that felt flawless measured 82.6 percent against production traffic, with the risk-bearing slices far below that.

![bloom ladder](assets/Figure_1_4-bloom.png)

The Bloom ladder above is how the chapter grades the risk: the higher the cognitive level you delegate, the wider the answer space and the larger the gap. Chapter 2 turns this diagnosis into an engineering posture: the Deterministic Shell around the probabilistic core.

## Exercises

1. **Reflection:** which slice of your own production traffic does your demo rehearse, and what share of real traffic does that slice actually carry?
2. **Application:** change the chain length in cell 3 to your own pipeline's step count and recompute the end-to-end rate. Find the step count at which you would no longer ship without a gate.
3. **Discussion:** the agent loop drifted in 10 out of 10 runs. Argue when that variability is acceptable, and what boundary you would add before letting the loop touch production state.